# 04 — Integration (Harmony), UMAP, Leiden clustering

**Input**: `03_normalized_hvg.h5ad`  
**Output**: `04_integrated_clustered.h5ad`

Steps:
1. PCA on HVGs (50 components)
2. Harmony batch correction by `Patient`
3. kNN graph on the Harmony-corrected embedding
4. UMAP for visualization
5. Leiden clustering at multiple resolutions

### Why Harmony and not BBKNN or scVI?

- **Harmony**: CPU-fast, linear embedding correction, well-validated, no GPU
- **BBKNN**: fast but modifies the kNN graph only, not the embedding
- **scVI**: arguably best quality but requires GPU/extra setup

For an 8 GB Mac with this dataset, Harmony is the right call.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import scanpy as sc
import matplotlib.pyplot as plt

from src import config as cfg
from src import preprocessing as pp

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 5))

adata = sc.read_h5ad(cfg.H5AD_NORM)
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')
print(f'HVGs: {adata.var["highly_variable"].sum()}')

In [ ]:
# Create subset of HVGs FIRST, then scale only those (much lighter on memory)
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10, zero_center=True)

# PCA on the subset (HGVs)
sc.tl.pca(adata_hvg, n_comps=cfg.N_PCS, random_state=cfg.RANDOM_SEED, svd_solver='arpack')

# Transfer PCA results to full adata; keep adata.X log-normalized
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.uns['pca'] = adata_hvg.uns['pca']

# Cleanup the temporary
del adata_hvg
import gc; gc.collect()
print("PCA done. PC count:", adata.obsm['X_pca'].shape[1])

In [ ]:
#Plot variance ratio: Inspect variance explained to confirm 50 PCs is reasonable
sc.pl.pca_variance_ratio(adata, n_pcs=cfg.N_PCS, log=True, show=False)
plt.savefig(cfg.FIGURES_DIR / 'pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Harmony integration
adata = pp.run_harmony(
    adata,
    batch_key=cfg.BATCH_KEY,
    theta=cfg.HARMONY_THETA,
    random_state=cfg.RANDOM_SEED,
)
print('obsm keys:', list(adata.obsm.keys()))

In [ ]:
import harmonypy as hm
print(f"harmonypy version: {hm.__version__}")

# Check the shape before transposing
ho = hm.run_harmony(
    adata.obsm["X_pca"],
    adata.obs,
    vars_use=[cfg.BATCH_KEY],
    theta=cfg.HARMONY_THETA,
    random_state=cfg.RANDOM_SEED,
    max_iter_harmony=20,
)
print(f"Z_corr shape: {ho.Z_corr.shape}")
print(f"Z_corr.T shape: {ho.Z_corr.T.shape}")
print(f"adata.n_obs: {adata.n_obs}")

In [ ]:
# Neighbors, UMAP, Leiden at multiple resolutions
adata = pp.run_neighbors_umap_leiden(
    adata,
    use_rep='X_pca_harmony',
    n_neighbors=cfg.N_NEIGHBORS,
    resolutions=cfg.LEIDEN_RESOLUTIONS,
    random_state=cfg.RANDOM_SEED,
)

In [ ]:
# Visualize: UMAP colored by batch (Patient), subtype, and Leiden clusters
# If patients mix well on UMAP = Harmony worked. If they separate = residual batch effect.
colorings = [cfg.BATCH_KEY, cfg.SUBTYPE_COLUMN, 'leiden_r0.8']
colorings = [c for c in colorings if c in adata.obs.columns]

sc.pl.umap(adata, color=colorings, wspace=0.6, show=False)
plt.savefig(cfg.FIGURES_DIR / 'umap_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Also color by authors' cell type annotations if present -- this is gold-standard
if cfg.CELLTYPE_COLUMN_AUTHORS in adata.obs.columns:
    sc.pl.umap(
        adata,
        color=cfg.CELLTYPE_COLUMN_AUTHORS,
        legend_loc='on data',
        legend_fontsize=7,
        show=False,
    )
    plt.savefig(cfg.FIGURES_DIR / 'umap_authors_celltypes.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# How many of each cell type per Leiden cluster?
import pandas as pd
ct = pd.crosstab(
    adata.obs['leiden_r0.5'],
    adata.obs['celltype_major']
)
# Show columns with epithelial types only
epi_cols = [c for c in ct.columns if 'Epithelial' in c]
print(ct[epi_cols].sort_values(epi_cols[0], ascending=False))

Cancer and normal epithelial cells overlap in some clusters, but others like cluster 4  and 5 are mainly cancer epithelial cells, while cluster 10 for ex. only has cancer cells.
I am running higher resolution to see if this would separate cancer from normal epit. cells (Leiden_0.8)

In [ ]:
ct_08 = pd.crosstab(
    adata.obs['leiden_r0.8'],
    adata.obs['celltype_major']
)
epi_cols = [c for c in ct_08.columns if 'Epithelial' in c]
print(ct_08[epi_cols][ct_08[epi_cols].sum(axis=1) > 50])

In [ ]:
#Check with resolution 1.0 to see if we can get more pure cancer clusters
ct_10 = pd.crosstab(
    adata.obs['leiden_r1.0'],
    adata.obs['celltype_major']
)
epi_cols = [c for c in ct_10.columns if 'Epithelial' in c]
mixed = ct_10[epi_cols]
mixed = mixed[mixed.sum(axis=1) > 20]
mixed['pct_cancer'] = (mixed['Cancer Epithelial'] / mixed.sum(axis=1) * 100).round(1)
print(mixed.sort_values('Cancer Epithelial', ascending=False))

Similar to leiden 0.8 so I'll use Leiden 0.8 and it matches Wu et al.

In [ ]:
# Sanity check: cluster counts across resolutions
import pandas as pd
res_counts = {
    f'leiden_r{r}': adata.obs[f'leiden_r{r}'].nunique()
    for r in cfg.LEIDEN_RESOLUTIONS
}
pd.Series(res_counts, name='n_clusters')

In [ ]:
adata.write_h5ad(cfg.H5AD_INTEGRATED, compression='gzip')
print(f'Saved: {cfg.H5AD_INTEGRATED} ({cfg.H5AD_INTEGRATED.stat().st_size / 1e6:.1f} MB)')

---

**Next**: `05_annotation.ipynb`